# 와인 품질 예측 심화 연구: 전처리 전략 및 최적화

## 1. 연구 개요 및 프로세스
본 연구는 와인 데이터셋의 품질(`quality`)을 예측하기 위해 **데이터 전처리 및 피처 엔지니어링의 최적화**에 집중합니다. 단순한 모델 적용을 넘어, 데이터의 통계적 특성을 고려한 엄밀한 전처리 과정을 거칩니다.

### 🔄 분석 프로세스 (Iterative Process)
1. **데이터 진단**: 결측치 및 기초 통계 확인
2. **이상치 제어 (Outlier Handling)**: PCA+IQR vs Isolation Forest 비교 및 선정
3. **스케일링 선정 (Scaler Selection)**: 다양한 스케일러 성능 비교
4. **특성 공학 (Iterative FE)**: 
    - 파생변수 생성 -> 다중공선성(VIF) 진단 -> 변수 제거 -> 재진단 (반복)
5. **최종 모델링**: 최적화된 데이터셋으로 회귀 vs 분류 성능 비교

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import warnings

from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.linear_model import LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.combine import SMOTETomek

from catboost import CatBoostRegressor, CatBoostClassifier

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

## 2. 데이터 로드 및 결측치 확인
가장 먼저 데이터의 누락된 값을 확인합니다.

In [ ]:
# 데이터 로드
try:
    df = pd.read_csv('wine_data_homework.csv', index_col=0)
except:
    # 데모용 데이터 생성 (실행 환경 고려)
    print("데이터 파일 없음. 데모 데이터를 생성합니다.")
    df = pd.DataFrame(np.random.rand(500, 11), columns=['fixed acidity', 'volatile acidity', 'citric acid',
       'residual sugar', 'chlorides', 'free sulfur dioxide',
       'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol'])
    df['quality'] = np.random.randint(3, 9, 500)

# 결측치 확인
missing_values = df.isnull().sum()
print("=== 결측치 확인 ===")
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("결측치가 없습니다. 다음 단계로 진행합니다.")
else:
    print("결측치가 발견되었습니다. 중앙값(Median)으로 대체합니다.")
    df.fillna(df.median(), inplace=True)

## 3. 이상치 제거 (Outlier Detection)
**Method A (PCA + IQR)**와 **Method B (Isolation Forest)**를 비교하여 더 합리적인 방법을 선택합니다.
와인 데이터에서 이상치는 '특이한 맛'을 내는 중요한 데이터일 수 있으므로, 데이터 손실을 최소화하면서 분포를 해치는 노이즈만 제거하는 것이 목표입니다.

In [ ]:
def compare_outlier_methods(data, target_col='quality'):
    X = data.drop(target_col, axis=1)
    y = data[target_col]
    
    # --- Method A: PCA + IQR ---
    # 차원을 축소하여 주요 변동성 방향에서의 이상치 탐지
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    Q1 = np.percentile(X_pca, 25, axis=0)
    Q3 = np.percentile(X_pca, 75, axis=0)
    IQR = Q3 - Q1
    
    # PC1, PC2 축 모두에서 벗어난 경우만 이상치로 간주 (보수적 접근)
    outlier_mask_pca = ((X_pca < (Q1 - 1.5 * IQR)) | (X_pca > (Q3 + 1.5 * IQR))).any(axis=1)
    
    # --- Method B: Isolation Forest ---
    # 다차원 공간에서 고립된 데이터 탐지
    iso = IsolationForest(contamination=0.02, random_state=42) # 2% 정도만 제거 목표
    outlier_pred_iso = iso.fit_predict(X)
    outlier_mask_iso = outlier_pred_iso == -1
    
    # 결과 비교
    n_outliers_pca = outlier_mask_pca.sum()
    n_outliers_iso = outlier_mask_iso.sum()
    
    print(f"Method A (PCA+IQR) 제거 수: {n_outliers_pca}")
    print(f"Method B (Isolation Forest) 제거 수: {n_outliers_iso}")
    
    # 시각화 비교
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=np.where(outlier_mask_pca, 'red', 'blue'), alpha=0.5, s=10)
    plt.title(f'PCA + IQR (Removed: {n_outliers_pca})')
    
    plt.subplot(1, 2, 2)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=np.where(outlier_mask_iso, 'red', 'blue'), alpha=0.5, s=10)
    plt.title(f'Isolation Forest (Removed: {n_outliers_iso})')
    
    plt.show()
    
    return outlier_mask_pca, outlier_mask_iso

mask_pca, mask_iso = compare_outlier_methods(df)

# 선정 로직: 이상치는 제거하되, 데이터 손실이 적고 다차원 관계를 고려하는 Isolation Forest 선택
# (사용자 판단에 따라 변경 가능, 여기서는 Isolation Forest 채택)
df_clean = df[~mask_iso].copy()
print(f"최종 데이터 크기: {df_clean.shape} (Isolation Forest 적용)")

## 4. 스케일링 기법 선정 (Scaler Selection)
이상치가 제거된 데이터에 대해 어떤 스케일러가 모델 성능에 가장 유리한지 테스트합니다.

In [ ]:
def select_best_scaler(data, target_col='quality'):
    X = data.drop(target_col, axis=1)
    y = data[target_col]
    
    scalers = {
        'Standard': StandardScaler(),
        'MinMax': MinMaxScaler(),
        'Robust': RobustScaler(),
        'MaxAbs': MaxAbsScaler()
    }
    
    best_score = float('inf')
    best_scaler_name = ''
    
    print("=== 스케일러 성능 비교 (RMSE 낮을수록 좋음) ===")
    # 빠른 비교를 위해 CatBoost Regressor 기본값 사용
    model = CatBoostRegressor(verbose=0, random_state=42)
    
    results = {}
    
    for name, scaler in scalers.items():
        X_scaled = scaler.fit_transform(X)
        # 3-Fold CV로 RMSE 측정
        scores = cross_val_score(model, X_scaled, y, scoring='neg_root_mean_squared_error', cv=3)
        rmse = -scores.mean()
        results[name] = rmse
        print(f"{name} Scaler RMSE: {rmse:.4f}")
        
        if rmse < best_score:
            best_score = rmse
            best_scaler_name = name
            
    print(f"\n>>> 선정된 스케일러: {best_scaler_name}")
    return scalers[best_scaler_name]

best_scaler = select_best_scaler(df_clean)

## 5. 반복적 피처 엔지니어링 (Iterative Feature Engineering)

**"확인 -> 적용 -> 재확인"**의 루프를 구현합니다.
1.  **VIF 확인**: 다중공선성 진단.
2.  **피처 제거**: 공선성이 높은 변수 제거.
3.  **파생변수 추가**: 도메인 지식 기반 변수 추가.
4.  **최종 선택**: 중요도가 낮은 변수 제거.

In [ ]:
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    return vif_data.sort_values(by="VIF", ascending=False)

target = 'quality'
X_fe = df_clean.drop(target, axis=1)
y_fe = df_clean[target]

# [1단계] 초기 VIF 확인
print("--- [1단계] 초기 VIF 확인 ---")
vif_initial = calculate_vif(X_fe)
display(vif_initial.head())

# [2단계] 높은 VIF 변수 처리 (예: VIF > 10)
# 주의: density와 alcohol은 와인에서 중요하지만 공선성이 높음. 
# 통계적으로는 제거해야 하나, 도메인 지식상 파생변수로 합치고 원본은 제거하는 전략 사용
print("\n--- [2단계] 파생 변수 생성 및 중복 변수 제거 ---")
X_fe['total_acidity'] = X_fe['fixed acidity'] + X_fe['volatile acidity']
X_fe['alcohol_density'] = X_fe['alcohol'] / X_fe['density'] # 알코올과 밀도 관계 결합

# 상관관계가 너무 높은 원본 변수 제거 (밀도, 산도 개별 변수 등)
drop_cols = ['density', 'fixed acidity'] # 예시: VIF가 높고 파생변수에 반영된 것들
X_fe = X_fe.drop(columns=drop_cols, errors='ignore')

# [3단계] VIF 재확인
print("\n--- [3단계] 수정 후 VIF 재확인 ---")
vif_final = calculate_vif(X_fe)
display(vif_final.head())

# [4단계] 상관관계 분석 및 최종 피처 선택
print("\n--- [4단계] 최종 피처 선택 (Correlation with Target) ---")
# 잠시 y를 붙여서 상관관계 확인
temp_df = X_fe.copy()
temp_df['target'] = y_fe
corr = temp_df.corr()['target'].drop('target').sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=corr.values, y=corr.index)
plt.title("Correlation with Quality (After FE)")
plt.show()

# 최종 데이터셋 확정
X_final = X_fe
y_final = y_fe
print(f"최종 피처 목록: {list(X_final.columns)}")

## 6. 최종 모델링 및 평가

선정된 전처리(Isolation Forest -> Best Scaler -> FE Features)를 적용하고, **회귀 vs 분류** 모델을 비교합니다.

In [ ]:
def regression_to_proba(y_pred, classes, alpha=2.0):
    n_samples = len(y_pred)
    n_classes = len(classes)
    probs = np.zeros((n_samples, n_classes))
    for i, cls in enumerate(classes):
        distance = (y_pred - cls) ** 2
        probs[:, i] = np.exp(-alpha * distance)
    return probs / probs.sum(axis=1)[:, np.newaxis]

def train_and_evaluate():
    classes = np.sort(y_final.unique())
    
    # Split
    X_train, X_val, y_train, y_val = train_test_split(X_final, y_final, test_size=0.2, stratify=y_final, random_state=42)
    
    # Scaling (선정된 Scaler 사용)
    X_train_s = best_scaler.fit_transform(X_train)
    X_val_s = best_scaler.transform(X_val)
    
    # SMOTE-Tomek
    smt = SMOTETomek(random_state=42)
    X_train_res, y_train_res = smt.fit_resample(X_train_s, y_train)
    
    print("\n=== 모델 성능 비교 ===")
    
    # 1. Regression Model
    reg = CatBoostRegressor(verbose=0, random_state=42, loss_function='RMSE')
    reg.fit(X_train_res, y_train_res)
    pred_reg = reg.predict(X_val_s)
    
    # Regression -> Class 변환
    pred_reg_class = np.clip(np.round(pred_reg), classes.min(), classes.max())
    acc_reg = accuracy_score(y_val, pred_reg_class)
    f1_reg = f1_score(y_val, pred_reg_class, average='macro')
    
    # Regression -> Proba 변환 (ROC AUC)
    proba_reg = regression_to_proba(pred_reg, classes)
    try:
        auc_reg = roc_auc_score(y_val, proba_reg, multi_class='ovr', average='macro')
    except: auc_reg = 0.5
    
    print(f"[Regression] Acc: {acc_reg:.4f}, F1(Macro): {f1_reg:.4f}, AUC: {auc_reg:.4f}")
    
    # 2. Classification Model
    clf = CatBoostClassifier(verbose=0, random_state=42, loss_function='MultiClass')
    clf.fit(X_train_res, y_train_res)
    pred_clf = clf.predict(X_val_s).flatten()
    proba_clf = clf.predict_proba(X_val_s)
    
    acc_clf = accuracy_score(y_val, pred_clf)
    f1_clf = f1_score(y_val, pred_clf, average='macro')
    try:
        auc_clf = roc_auc_score(y_val, proba_clf, multi_class='ovr', average='macro')
    except: auc_clf = 0.5
    
    print(f"[Classification] Acc: {acc_clf:.4f}, F1(Macro): {f1_clf:.4f}, AUC: {auc_clf:.4f}")

    # 시각화
    metrics = pd.DataFrame({
        'Model': ['Regression', 'Classification'],
        'Accuracy': [acc_reg, acc_clf],
        'F1-Macro': [f1_reg, f1_clf]
    })
    metrics.plot(x='Model', kind='bar', figsize=(8, 5), title='Final Performance Comparison')
    plt.show()

train_and_evaluate()